In [ ]:
pip install scikit-surprise

In [ ]:
import pandas as pd

data = pd.read_csv("/content/cleaned_ecommerce_dataset.csv")

print(data.head())
print("Dataset shape:", data.shape)


                            user_id                           item_id  \
0  98758d88bf4b8eef1372ddee45d63178  fe59a1e006df3ac42bf0ceb876d70969   
1  87ae4c644c15d9c6b6f826dfec33b340  e19ddcc85537b41f22116c8d5425ef46   
2  2e875ea57961ad115cec13fef0920ae6  880be32f4db1d9f6e2bec38fb6ac23ab   
3  3c857a6f7828bfb70fb712e2393cfd1b  1f9799a175f50c9fa725984775cac5c5   
4  3c857a6f7828bfb70fb712e2393cfd1b  13944d17b257432717fd260e69853140   

                 category   price  freight_value  customer_city  \
0  informatica_acessorios  809.10          44.29   campo alegre   
1        moveis_decoracao   29.99          15.10  volta redonda   
2              brinquedos   44.90           7.16   porto alegre   
3         cama_mesa_banho   59.90           9.94      sao paulo   
4         cama_mesa_banho   59.90           9.94      sao paulo   

  customer_state  interaction  purchase_count  
0             AL            1               1  
1             RJ            1               1  
2             

In [ ]:
from surprise import Dataset, Reader

In [ ]:
# Define rating scale using purchase_count
reader = Reader(
    rating_scale=(1, int(data['purchase_count'].max()))
)

dataset = Dataset.load_from_df(
    data[['user_id', 'item_id', 'purchase_count']],
    reader
)

In [ ]:
from surprise.model_selection import train_test_split

trainset, testset = train_test_split(
    dataset,
    test_size=0.2,
    random_state=42
)

In [ ]:
from surprise import SVD

In [ ]:
svd_model = SVD(
    n_factors=50,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

svd_model.fit(trainset)

print("SVD model trained successfully")


SVD model trained successfully


In [ ]:
from surprise import accuracy

predictions = svd_model.test(testset)

rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

RMSE: 0.0828
MAE:  0.0241


In [ ]:
def recommend_products_svd(model, data, user_id, N=5):
    all_items = data['item_id'].unique()

    predictions = [
        (item, model.predict(user_id, item).est)
        for item in all_items
    ]

    predictions.sort(key=lambda x: x[1], reverse=True)

    return [item for item, _ in predictions[:N]]


In [ ]:
sample_user = data['user_id'].iloc[0]

recommendations = recommend_products_svd(
    svd_model,
    data,
    sample_user,
    N=5
)

# Create a mapping from item_id to category
product_to_category = data.set_index('item_id')['category'].to_dict()

recommended_with_categories = []

for pid in recommendations:
    category = product_to_category.get(pid, "Unknown Category")
    recommended_with_categories.append(f"{pid} ({category})")
print("User:", sample_user)
print("Recommended products with categories:")

for product in recommended_with_categories:
    print(product)


User: 98758d88bf4b8eef1372ddee45d63178
Recommended products with categories:
72172e982e8b92155069e4201c92c0bb (esporte_lazer)
00faa46f36261af8bbf3a4d37fa4841b (fashion_bolsas_e_acessorios)
7f064525eaaa1ce9d22c085f7ff5413a (relogios_presentes)
113f80f12c8892f0c59206f70b862b40 (cama_mesa_banho)
ce066f4a83649651549d717c9b566816 (esporte_lazer)
